In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [2]:
df = pd.read_csv("diabetes.csv")
print(df.head())

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [4]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [5]:
mm = MinMaxScaler()
df['Glucose'] = mm.fit_transform(df[['Glucose']])
df['BMI'] = mm.fit_transform(df[['BMI']])
cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Replace 0s with NaN for medically invalid zeros
df[cols] = df[cols].replace(0, np.nan)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,0.743719,72.0,35.0,NaN,0.500745,0.627,50,1
1,1,0.427136,66.0,29.0,NaN,0.396423,0.351,31,0
2,8,0.919598,64.0,NaN,NaN,0.347243,0.672,32,1
3,1,0.447236,66.0,23.0,94.0,0.418778,0.167,21,0
4,0,0.688442,40.0,35.0,168.0,0.642325,2.288,33,1


In [6]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [7]:
print(X.shape, y.shape)

(768, 8) (768,)


In [8]:
# BUG FIX: was 'X_text' (typo) — corrected to 'X_test'
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
# Impute missing values using median strategy
# fit_transform on train, transform only on test (avoid data leakage)
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (614, 8)
Test shape: (154, 8)


In [10]:
# GridSearchCV to find best SVM hyperparameters
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']
}

svc = SVC(random_state=42)
grid_search = GridSearchCV(svc, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy: {:.2f}%".format(grid_search.best_score_ * 100))

Best Parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
Best Cross-Validation Accuracy: 78.02%


In [18]:
# Evaluate on test set
# y_pred = grid_search.predict(X_test)

# print("Test Accuracy: {:.2f}%".format(accuracy_score(y_test, y_pred) * 100))
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))
# print("Confusion Matrix:")
# print(confusion_matrix(y_test, y_pred))

In [19]:
best_model = grid_search.best_estimator_
ypred_acc = best_model.predict(X_test)

In [20]:
print("\nAccuracy Score: ")
print(accuracy_score(y_test, ypred_acc))
print("\nClassification Report: ")
print(classification_report(y_test, ypred_acc))
print("\nConfusion Matrix: ")
print(confusion_matrix(y_test, ypred_acc))  # removed the quotes


Accuracy Score: 
0.7012987012987013

Classification Report: 
              precision    recall  f1-score   support

           0       0.74      0.83      0.78       100
           1       0.60      0.46      0.52        54

    accuracy                           0.70       154
   macro avg       0.67      0.65      0.65       154
weighted avg       0.69      0.70      0.69       154


Confusion Matrix: 
[[83 17]
 [29 25]]
